# RSNA Knee — competition-data-only offline inference
This notebook runs the attached, trained image model against the official test.csv. It does not train, access radiology reports, call a hosted API, download assets, or submit predictions.
The existing account operator must attach the official competition data, this complete source, selected trained checkpoint and any preverified decoder wheels; set Internet **Off** and the same profiled accelerator. No performance or nine-hour compliance is claimed until the actual full run finishes.

In [ ]:
import time
NOTEBOOK_START = time.monotonic()  # Includes dependency setup and model/data loading.
from pathlib import Path
import sys
import subprocess
import json


In [ ]:
# Pin the EXISTING trained candidate. Never select by the newest filename.
DATA_ROOT = Path('/kaggle/input/rsna-knee-abnormality-detection')
SOURCE_DIR = None  # Set to attached directory containing rsna_infer.py, rsna_data.py, rsna_image_model.py.
CHECKPOINT = None  # Set to the selected trained .pt file inside /kaggle/input.
CHECKPOINT_SHA256 = ''  # Exact SHA-256 from the successful training receipt.
WHEEL_DIR = None  # Optional attached offline wheelhouse already verified on this Kaggle image.
DEVICE = 'cuda'
WORKERS = 2
if SOURCE_DIR is None or CHECKPOINT is None or len(CHECKPOINT_SHA256) != 64:
    raise RuntimeError('Attach and pin the selected trained model/source before running; no placeholder predictions are allowed.')
DATA_ROOT, SOURCE_DIR, CHECKPOINT = DATA_ROOT.resolve(), Path(SOURCE_DIR).resolve(), Path(CHECKPOINT).resolve()
for path in (DATA_ROOT, SOURCE_DIR, CHECKPOINT):
    if not path.is_relative_to('/kaggle/input') or not path.exists():
        raise RuntimeError('Every input must be an existing attached /kaggle/input asset.')


In [ ]:
# Optional installation is local-only. A missing compatible wheel fails; no network fallback.
if WHEEL_DIR is not None:
    WHEEL_DIR = Path(WHEEL_DIR).resolve()
    if not WHEEL_DIR.is_relative_to('/kaggle/input') or not WHEEL_DIR.is_dir():
        raise RuntimeError('Wheelhouse must be an attached input directory.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--disable-pip-version-check',
                    '--find-links', str(WHEEL_DIR), 'pydicom>=3.0', 'pylibjpeg',
                    'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg'], check=True)
import torch, numpy, PIL, pydicom
if tuple(int(part) for part in pydicom.__version__.split('.')[:2]) < (3, 0):
    raise RuntimeError('This source requires pydicom >= 3.0.')
sys.path.insert(0, str(SOURCE_DIR))
from rsna_infer import run_inference


In [ ]:
receipt = run_inference(data_root=DATA_ROOT, checkpoint=CHECKPOINT,
    checkpoint_sha256=CHECKPOINT_SHA256, output_dir='/kaggle/working',
    device=DEVICE, batch_size=1, workers=WORKERS, start_monotonic=NOTEBOOK_START)
# Aggregate result only. Never display test IDs, images, report text or a CSV head.
print(json.dumps({key: receipt[key] for key in ('status', 'rows', 'elapsed_seconds',
                  'checkpoint_sha256', 'submission_sha256')}, indent=2))
assert Path('/kaggle/working/submission.csv').is_file()
